# Outline

* Required Installation
* QR-PUF
    - BB84 Qubit Encoding
* XOR-APUF (CPUF) Implementation
* BB84-Encoded PUF Responses
    - Threat Model
    - From One-Copy Quantum Measurement to Label Noise
* Modeling Attack (`LRAttack2021`) for HPUF
* Runtime Patches for `LRAttack2021`
* Bonus: Qiskit Implementation
    - Setup
    - BB84 State-Preparation Circuits
    - Eve's Optimal Measurement — the Breidbart Basis
    - Running the Circuit Simulation and Verifying the Analytical Model
    - Modeling Attack (`LRAttack2021`) on the Qiskit-Simulated HPUF
    - Attack Helper
    - Running the Attack

## 1. Required Installation

Run the cell below once to install the required packages (the same set as Lab 1); if they are already installed, you can skip this step:

- `pypuf==3.2.1` — PUF simulations (`XORArbiterPUF`) and the `LRAttack2021` modeling attack
- `tensorflow` — the training backend used by `LRAttack2021`
- `numpy<2.0` — pypuf 3.2.1 is not compatible with NumPy 2.x
- `matplotlib` — plotting

Qiskit is only needed for the bonus Section 7 and is installed there.

In [ ]:
%pip install pypuf==3.2.1
%pip install tensorflow "numpy<2.0" matplotlib


## 2. QR-PUF

As we saw in Lab 1, XOR Arbiter PUFs (XOR-APUFs) are physically unclonable but **mathematically
modelable**: given enough challenge-response pairs (CRPs), an attacker (Eve) can build a software
clone with high prediction accuracy using `LRAttack2021`. The reason is the additive delay model:
each of the $k$ components is a linear threshold function of the $\phi$-parity features, so the whole
PUF can be learned as a product of $k$ linear models. (For $k=1$ the PUF is linearly separable in
$\phi$-space; for $k>1$ it is not, but it remains learnable given enough CRPs.)

This raises a question: *what if the response isn't handed to the attacker as an observable
classical bit?*

The **Quantum-Readout PUF (QR-PUF)**, proposed by Škorić, challenges a physical (optical) PUF directly
with **quantum states**, so that an eavesdropper cannot read the challenge or the response without
disturbing them. The **Hybrid PUF (HPUF)** from the Quantum Lock paper (arXiv:2110.09469), used in
this lab, follows the same idea of hiding the response inside quantum states, but keeps the PUF
itself **classical** and only encodes its output:

$$
\text{Classical PUF output} \;\xrightarrow{\text{BB84 encoding}}\; \text{Non-orthogonal quantum state}
$$

(The same paper's **Hybrid Locked PUF (HLPUF)** adds a locking mechanism on top of HPUF against
adaptive adversaries; it is out of scope for this lab.)

Two properties of quantum mechanics make this useful:

1. **No-cloning theorem** — an unknown quantum state can't be perfectly copied, so Eve can't both
   forward the qubit to the verifier and keep a copy for her own analysis.
2. **Indistinguishability of non-orthogonal states** — if the encoding basis ($Z$ or $X$) is randomly
   chosen, measuring a single copy in the wrong basis returns a **completely random bit**, unrelated
   to the original value.

### 2.1 BB84 Qubit Encoding

BB84 (Bennett & Brassard, 1984) is normally a key-distribution protocol; here we only use its
**encoding scheme** — turning a classical bit pair into an unclonable quantum state.

| Basis | Symbol | 0 → State | 1 → State |
|---|---|---|---|
| Rectilinear ($Z$) | $\{\lvert 0\rangle, \lvert 1\rangle\}$ | $\lvert 0\rangle$ | $\lvert 1\rangle$ |
| Diagonal ($X$) | $\{\lvert +\rangle, \lvert -\rangle\}$ | $\lvert +\rangle = \frac{\lvert 0\rangle + \lvert 1\rangle}{\sqrt{2}}$ | $\lvert -\rangle = \frac{\lvert 0\rangle - \lvert 1\rangle}{\sqrt{2}}$ |

These four states are **non-orthogonal** across the two bases — the cornerstone of the scheme's
security. Each qubit carries two classical bits:

- **basis bit** — which basis was used (0 → $Z$, 1 → $X$)
- **value bit** — which state within that basis (0 or 1)

The PUF's (basis, value) pairs map to a qubit as:

$$
\lvert \psi \rangle =
\begin{cases}
\lvert 0 \rangle & \text{basis}=0,\ \text{value}=0 \\
\lvert 1 \rangle & \text{basis}=0,\ \text{value}=1 \\
\lvert + \rangle & \text{basis}=1,\ \text{value}=0 \\
\lvert - \rangle & \text{basis}=1,\ \text{value}=1
\end{cases}
$$

The preparation basis is physically invisible to whoever receives the qubit. Measuring in the
**correct** basis recovers the value bit with certainty; measuring in the **wrong** basis gives a
uniformly random bit, carrying no information about the original value.

A naive strategy — guess the basis, then
measure — gets the basis right only half the time, and a wrong-basis measurement is fully random, so
it flips the true response on average **25%** of the time. Section 4.2 derives the *optimal*
single-copy strategy, which does better (~14.64% error), and Section 7 verifies it with a circuit
simulation of Eve's measurement.

## 3. XOR-APUF (CPUF) Implementation

The following code implements the XOR-APUF used as the CPUF in this lab (see Lab 1 for details):

In [1]:
import numpy as np
from pypuf.simulation import XORArbiterPUF
from pypuf.io import random_inputs

n = 64
k = 5
puf = XORArbiterPUF(n=n, k=k, seed=1, noisiness=0.0)

train_N = 200000
test_N  = int(train_N * 0.2)   # size of a separate, fresh test set
train_challenges = random_inputs(n=n, N=train_N, seed=2)
train_responses = puf.eval(train_challenges)
test_challenges = random_inputs(n=n, N=test_N, seed=5)   # different seed: independent of the training challenges
test_responses = puf.eval(test_challenges)


## 4. BB84-Encoded PUF Responses

For each challenge $c_i$, the CPUF produces a basis bit $b_i=f_b(c_i)$ and a value bit
$\theta_i=f_\theta(c_i)$; the HPUF would encode these as the BB84 state $|\psi_{(b_i,\theta_i)}\rangle$.

### 4.1 Threat Model

The adversary receives $D_Q = \{(c_i, |\psi_{(b_i,\theta_i)}\rangle)\}_{i=1}^{q}$, and:

- knows the PUF architecture and the BB84 encoding rule, but not the hidden parameters of $f_b$/$f_\theta$;
- gets exactly **one** copy of each quantum response — no repeated measurements of the same state;
- has arbitrary (single-copy) measurement capability;
- wants a classical ML model that predicts the true response bit on unseen challenges.

The one-copy restriction is essential: with multiple copies of the same state, the adversary could
improve discrimination beyond the fixed error rate derived next.

### 4.2 From One-Copy Quantum Measurement to Label Noise

Eve wants the **value bit** $\theta$, but she doesn't know the **basis bit** $b$; from her point of view
$b$ is uniformly random. Averaging over $b$: if $\theta=0$, the emitted state is $|0\rangle$ ($b=0$) or
$|+\rangle$ ($b=1$) with equal probability; if $\theta=1$, it's $|1\rangle$ or $|-\rangle$. The
adversary must therefore distinguish the two mixed states:

$$
\rho_0 = \tfrac12\big(|0\rangle\langle 0| + |+\rangle\langle +|\big), \qquad
\rho_1 = \tfrac12\big(|1\rangle\langle 1| + |-\rangle\langle -|\big)
$$

By the Helstrom bound, the optimal single-copy success probability for two equally likely states is
$p_{guess}=\tfrac12+\tfrac14\lVert\rho_0-\rho_1\rVert_1$. Here $\rho_0-\rho_1=\tfrac12(Z+X)$ has
eigenvalues $\pm\tfrac{1}{\sqrt2}$, so $\lVert\rho_0-\rho_1\rVert_1=\sqrt2$ and:

$$
p_{guess} = \tfrac12 + \tfrac{1}{2\sqrt{2}} \approx 0.853553, \qquad
p_{error} = 1 - p_{guess} \approx 0.146447
$$

This ~14.64% isn't physical noise from the PUF or from BB84 preparation — it's the effective
classical label-error rate after the *optimal* one-copy measurement. So instead of simulating the
measurement itself, we can generate HPUF responses directly from this known statistic:

$$
\theta \rightarrow |\psi_{(b,\theta)}\rangle \rightarrow \text{optimal measurement} \rightarrow \tilde{\theta}, \qquad
\Pr[\tilde{\theta} = \theta] = 0.853553,\ \ \Pr[\tilde{\theta} \neq \theta] = 0.146447
$$

In [2]:
clean_responses = train_responses.copy()

p_guess = 0.5 + 1/(2*np.sqrt(2))
p_error = 1 - p_guess
print('p_guess =', p_guess)
print('p_error =', p_error)

rng = np.random.default_rng(12345)
measured_responses = clean_responses.copy()
flip = rng.random(len(measured_responses)) < p_error
measured_responses[flip] *= -1

observed_error = np.mean(
    measured_responses.reshape(-1) != clean_responses.reshape(-1)
)
print('Observed extraction error:', observed_error)

p_guess = 0.8535533905932737
p_error = 0.14644660940672627
Observed extraction error: 0.14664


For a large training database, the observed extraction error should be close to 0.1464.

## 5. Modeling Attack (`LRAttack2021`) for HPUF

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from pypuf.simulation.base import LTFArray
from pypuf.io import random_inputs, ChallengeResponseSet
from pypuf.attack import LRAttack2021
import pypuf.attack

quantum_crps = ChallengeResponseSet(
    train_challenges[:train_N],
    measured_responses[:train_N]
)
attack_q = LRAttack2021(
    quantum_crps,
    seed=100,
    k=k,
    bs=1000,
    lr=0.001,
    epochs=50,
    stop_validation_accuracy=1.0
)
model_q = attack_q.fit()

predicted_q = model_q.eval(test_challenges)
accuracy_q = np.mean(
    predicted_q.reshape(-1) == test_responses.reshape(-1)
)
print('BB84-interface attack accuracy:', accuracy_q)

Epoch 1/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.4991 - loss: 0.6932 - val_accuracy: 0.4965 - val_loss: 0.6931
Epoch 2/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 994us/step - accuracy: 0.5039 - loss: 0.6931 - val_accuracy: 0.5110 - val_loss: 0.6932
Epoch 3/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 975us/step - accuracy: 0.5042 - loss: 0.6928 - val_accuracy: 0.5090 - val_loss: 0.6932
Epoch 4/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 964us/step - accuracy: 0.5069 - loss: 0.6925 - val_accuracy: 0.5025 - val_loss: 0.6934
Epoch 5/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 991us/step - accuracy: 0.5097 - loss: 0.6923 - val_accuracy: 0.4995 - val_loss: 0.6937
Epoch 6/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 989us/step - accuracy: 0.5105 - loss: 0.6921 - val_accuracy: 0.4995 - val_loss: 0.6940
Epoch 7/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step - accuracy: 0.5129 - loss: 0.6920 - val_accuracy: 0.4990 - val_loss: 0.6942
Epoch 8/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 957us/step - accuracy: 0.5131 - loss: 0.6

## 6. Runtime Patches for `LRAttack2021`

If the previous cell raised:

```
Traceback (most recent call last):
  File "../QPUF Tutorial/deneme.py", line 27, in <module>
    model = attack.fit()
  File "../.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 145, in fit
    self._history = model.fit(
  File "../.local/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "../.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 89, in accuracy
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)
TypeError: Expected int8, but got 0.5 of type 'float'.
```

run the cell below, then re-run the attack cell above.

In [4]:
def keras_to_pypuf_fixed(self, keras_model):
    n, k = self.crps.challenges.shape[1], self.k
    weights = np.zeros(shape=(k, n))
    bias = np.zeros(shape=(k,))
    for l in range(k):
        layer_weights = keras_model.layers[l + 1].get_weights()
        weights[l] = layer_weights[0][:, 0]
        bias[l] = layer_weights[1][0]
    return LTFArray(weight_array=weights, bias=bias, transform=XORArbiterPUF.transform_atf)
pypuf.attack.LRAttack2021.keras_to_pypuf = keras_to_pypuf_fixed

@staticmethod
def loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.losses.binary_crossentropy(.5 - .5 * y_true, .5 - .5 * y_pred)

@staticmethod
def accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)

pypuf.attack.LRAttack2021.loss = loss
pypuf.attack.LRAttack2021.accuracy = accuracy

## 7. Bonus: Qiskit Implementation

Sections 4-6 modeled the BB84 measurement **analytically**: instead of preparing and measuring real
qubits, we flipped each response bit with probability $p_{error}$. That shortcut is exact in
expectation, but this section builds the real thing — single-qubit BB84 circuits, Eve's optimal
measurement on Qiskit's `AerSimulator`, and a check that the empirical bit-flip rate matches
$p_{error}\approx 0.1464$.

It doesn't change anything computed above; it only requires the cells above to have run (including
the Section 6 patch cell, if your environment needed it).

### 7.1 Setup

Qiskit and its Aer simulator backend are not required by the rest of the notebook, so they are
installed separately here. `qiskit`/`qiskit-aer` may try to pull in a newer `numpy` than the
`pypuf`/`tensorflow` stack expects, so the `numpy<2.0` pin is passed in the same command: pip then
resolves Qiskit against the pinned version instead of upgrading `numpy`.

In [ ]:
%pip install qiskit qiskit-aer "numpy<2.0"


### 7.2 BB84 State-Preparation Circuits

Recall the encoding from Section 2.1 (in the notation of Section 4): a **value bit** $\theta \in \{0,1\}$ (the PUF response bit we
want to protect) is encoded together with a **basis bit** $b\in\{0,1\}$ into one of four
non-orthogonal single-qubit states. Both bits are PUF-derived ($b_i = f_b(c_i)$,
$\theta_i = f_\theta(c_i)$): $\theta$ comes from the CPUF whose responses we're protecting, and $b$
comes from a second, independent CPUF instance playing the role of $f_b$. Using a second PUF (rather
than a public coin flip) means the basis itself is also a PUF secret — an adversary can't query it
directly, only infer it (imperfectly) from the qubit it helped prepare.

| basis $b$ | value $\theta$ | state |
|---|---|---|
| 0 (rectilinear) | 0 | $\lvert 0\rangle$ |
| 0 (rectilinear) | 1 | $\lvert 1\rangle$ |
| 1 (diagonal)    | 0 | $\lvert +\rangle$ |
| 1 (diagonal)    | 1 | $\lvert -\rangle$ |

As a circuit, starting from $\lvert 0\rangle$: apply `X` if $\theta=1$ (selects $\lvert1\rangle$
over $\lvert0\rangle$), then apply `H` if $b=1$ (rotates into the conjugate
$\{\lvert+\rangle,\lvert-\rangle\}$ basis). The basis bit isn't something Eve can read off the
qubit itself — it only determines which state gets prepared.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

SIM = AerSimulator()

def build_bb84_circuits(basis_bits, value_bits):
    """One 1-qubit state-preparation circuit per (basis, value) pair, following the
    encoding table above: X if value=1, then H if basis=1."""
    circuits = []
    for basis, value in zip(basis_bits, value_bits):
        qc = QuantumCircuit(1, 1)
        if value == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits

### 7.3 Eve's Optimal Measurement — the Breidbart Basis

Eve doesn't know $b$, and gets only a **single copy** of each qubit. The best she can do is measure
in a *fixed* basis that maximizes her average chance of recovering $\theta$, regardless of which
encoding basis was actually used. That optimal fixed basis is the **Breidbart basis** — exactly
halfway between rectilinear ($Z$) and diagonal ($X$), at $22.5^\circ = \pi/8$. (This is the angle
between state vectors in Hilbert space; on the Bloch sphere, where the $Z$ and $X$ axes are $90^\circ$
apart, the Breidbart axis lies $45^\circ$ from each.)

We implement it by rotating the qubit with $R_Y(-\pi/4)$ — a $-45^\circ$ rotation on the Bloch sphere,
i.e. $-\pi/8$ in Hilbert space — which maps the Breidbart basis onto the computational $Z$ axis, and
then measuring in the computational basis. This reproduces the closed-form
result from Section 4.2:

$$
p_{guess} = \cos^2(\pi/8) = \tfrac12 + \tfrac{1}{2\sqrt2} \approx 0.853553, \qquad
p_{error} = \sin^2(\pi/8) = \tfrac12 - \tfrac{1}{2\sqrt2} \approx 0.146447.
$$

In [ ]:
def eve_breidbart_measurement(circuits):
    measured = []
    for qc in circuits:
        c = qc.copy()
        c.ry(-np.pi / 4, 0)
        c.measure(0, 0)
        measured.append(c)
    result = SIM.run(measured, shots=1, memory=True).result()
    guesses = np.array([int(result.get_memory(i)[0]) for i in range(len(circuits))])
    return guesses

### 7.4 Running the Circuit Simulation and Verifying the Analytical Model

We take the CPUF's clean training responses (`clean_responses`) as the value bits
$\theta_i=f_\theta(c_i)$, and derive the basis bits $b_i=f_b(c_i)$ from a second, independent
`XORArbiterPUF` instance (`puf_basis`) queried on the same challenges — matching the $f_b$/$f_\theta$
model from Section 4, rather than a public coin flip. We build the corresponding BB84 circuits, run
Eve's Breidbart-basis measurement through `AerSimulator` over the full training set, and compare her
empirical error rate against the theoretical $p_{error}\approx 0.1464$. The resulting database
(`measured_responses_qiskit`) is reused below for the `LRAttack2021` attack, so the circuits are only
simulated once.

In [ ]:
puf_basis = XORArbiterPUF(n=n, k=k, seed=101, noisiness=0.0)  # f_b: a second, independent CPUF instance

value_bits = ((1 - clean_responses.reshape(-1)) // 2).astype(int)                       # theta_i = f_theta(c_i)
basis_bits = ((1 - puf_basis.eval(train_challenges).reshape(-1)) // 2).astype(int)       # b_i     = f_b(c_i)

circuits = build_bb84_circuits(basis_bits, value_bits)
eve_guess = eve_breidbart_measurement(circuits)

empirical_error = np.mean(eve_guess != value_bits)
print(f'Theoretical p_error:  {p_error:.6f}')
print(f'Empirical p_error:    {empirical_error:.6f}   (N={train_N} simulated qubits)')

# back to pypuf's {-1,1} bipolar convention — this is Eve's attack database, reused below
measured_responses_qiskit = (1 - 2 * eve_guess).reshape(clean_responses.shape).astype(clean_responses.dtype)

**Takeaway.** The circuit-simulated bit-flip rate matches the closed-form $p_{error}\approx 0.1464$
used to build `measured_responses` earlier in the notebook, within the expected statistical
fluctuation for this sample size. This confirms the analytical shortcut used above is not an
approximation of convenience — it is exactly what an actual BB84 state-preparation-and-measurement
circuit produces under Eve's optimal single-copy strategy.

### 7.5 Modeling Attack (`LRAttack2021`) on the Qiskit-Simulated HPUF

Section 7.4 confirmed the *statistics* of the analytical shortcut match a real BB84 circuit. Here we
go one step further: attack the full quantum-circuit-derived noisy database (one real Qiskit
state-preparation-and-measurement per training challenge, not a Bernoulli coin flip) with the same
`LRAttack2021` attack used in Section 5. This is the closest this notebook gets to attacking an
actual BB84-encoded HPUF end-to-end: PUF $\to$ qubit $\to$ Eve's Breidbart-basis measurement
$\to$ classical ML attack.

### 7.6 Attack Helper

`run_lrattack` wraps a **single** `LRAttack2021` fit so it can be called the same way on different
CRP databases. It clears the Keras session first so repeated calls in this notebook don't accumulate
memory.

It relies on the runtime patches from Section 6 (`keras_to_pypuf`, `loss`, `accuracy`) if your
environment needed them. Patches stay active in the kernel once that cell has run; after a kernel
restart, re-run the Section 6 patch cell before this section.

Note: modeling a $k$-XOR Arbiter PUF is a non-convex problem, so a single fit can occasionally
converge to a poor local optimum (near-chance accuracy) instead of a good one. If a run looks stuck,
re-run the attack cell with a different `seed`.

In [ ]:
def run_lrattack(crps, k, seed=100, bs=1000, lr=0.001,
                 epochs=50, stop_validation_accuracy=1.0):
    """Fit LRAttack2021 once (single fit, single seed). Clears the Keras session first so repeated
    calls in this notebook don't accumulate memory (see the note above on non-convexity)."""
    tf.keras.backend.clear_session()
    attack = LRAttack2021(
        crps, seed=seed, k=k, bs=bs, lr=lr, epochs=epochs,
        stop_validation_accuracy=stop_validation_accuracy,
    )
    model = attack.fit()
    val_acc = attack.history['val_accuracy'][-1]
    print(f'final val_accuracy={val_acc:.4f}')
    return model, attack


### 7.7 Running the Attack

`measured_responses_qiskit` holds Eve's classical database as it actually came out of the Qiskit
circuit simulation. We train `LRAttack2021` on it, then evaluate the resulting model on
`test_challenges` — comparing its predictions against `test_responses`, the CPUF's true, noise-free
response, not a second noisy quantum measurement of the test challenges.

This is the right comparison: what Eve needs to forge is the same value $\theta=f_\theta(c)$ that a
legitimate verifier recovers deterministically once the correct basis is known — that's the whole
point of the encoding, only an adversary *without* the basis suffers the $p_{error}\approx0.1464$
measurement noise. Scoring Eve's clone against a second noisy measurement instead would measure how
well the model fits Eve's own measurement noise, not whether the clone can actually pass
verification.

In [ ]:
crps_qiskit = ChallengeResponseSet(train_challenges, measured_responses_qiskit)

print('Qiskit-circuit HPUF attack:')
model_qiskit, attack_qiskit = run_lrattack(crps_qiskit, k=k)

predicted_qiskit = model_qiskit.eval(test_challenges)
accuracy_qiskit = np.mean(
    predicted_qiskit.reshape(-1) == test_responses.reshape(-1)
)
print('Qiskit-circuit HPUF attack accuracy:', accuracy_qiskit)

**Takeaway.** This accuracy should land close to the `BB84-interface attack accuracy` computed
earlier in the notebook from the analytical shortcut — the two databases are statistically
equivalent (same $p_{error}\approx0.1464$ label-noise process), just produced by different means:
one from a closed-form coin flip, this one from an actual simulated BB84 state preparation and
Breidbart-basis measurement circuit run through Qiskit's `AerSimulator`.